# Phase 3.5 — Epoch Sensitivity Check

Quick exploration: at epoch 15 the loss was still dropping. Are we undertrained?

This notebook trains ONE fold (S1+S2 / all10 / test=2020) for 40 epochs,
evaluating on the test set at checkpoints (epochs 5, 10, 15, 20, 25, 30, 35, 40)
and tracking per-phase MAE.

If MAE keeps improving past epoch 15, we're undertrained and should rerun the
full grid with more epochs. If MAE plateaus before epoch 15, we're fine.

This takes ~5–10 min on the MIG A100 slice (one fold × 40 epochs).


## 1. Setup

In [1]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.baseline_lstm import (MultiStreamLSTM, soft_label_loss,
                                              STREAM_CONFIGS)
from phase3_baselines.evaluate import evaluate_predictions

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('GPU mem:', torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 'n/a', 'GB')

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'


Device: cuda
GPU mem: 5.100273664 GB


## 2. Load one fold (S1+S2 / all10 / test=2020)

In [2]:
YEARS = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR = 2020
policy = SelectionPolicy(use_only_rank1=False)  # all10

print('Loading test set...')
test_b = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{TEST_YEAR}.npz'), policy)
print(f'  test N={len(test_b["xid"])}')

print('Loading train set (4 years)...')
train_parts = {}
for y in YEARS:
    if y == TEST_YEAR: continue
    train_parts[y] = apply_selection(load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != TEST_YEAR])
del train_parts
gc.collect()
print(f'  train N={len(train_b["xid"])}')
phase_ids = train_b['phase_ids']


Loading test set...
  test N=5181
Loading train set (4 years)...
  train N=21634


## 3. Train for 40 epochs with checkpointed evaluation

In [3]:
# Use S1+S2 (our best stream from the grid)
STREAM = 'S1+S2'
active = STREAM_CONFIGS[STREAM]
HIDDEN = 32
BATCH_SIZE = 32
LR = 1e-3
TOTAL_EPOCHS = 40
CHECKPOINT_EPOCHS = [5, 10, 15, 20, 25, 30, 35, 40]

# Pre-build tensor bundles (avoid repeated from_numpy)
class TensorBundle:
    def __init__(self, bundle):
        self.X    = torch.from_numpy(bundle['X'])
        self.M    = torch.from_numpy(bundle['mask'])
        self.Y    = torch.from_numpy(bundle['Y'])
        self.LM   = torch.from_numpy(bundle['label_mask'])
        self.bag  = torch.from_numpy(bundle['bag_id'])
        self.rank = torch.from_numpy(bundle['rank'])
        self.yr   = torch.from_numpy(bundle['harvest_year'])
        self.xid  = torch.from_numpy(bundle['xid'])
        self.N    = self.X.shape[0]

tb_tr = TensorBundle(train_b)
tb_te = TensorBundle(test_b)

def iter_batches(tb, batch_size, shuffle, device, generator=None):
    idx = torch.randperm(tb.N, generator=generator) if shuffle else torch.arange(tb.N)
    for i in range(0, tb.N, batch_size):
        sl = idx[i:i+batch_size]
        x = tb.X[sl].to(device, non_blocking=True)
        m = tb.M[sl].to(device, non_blocking=True).to(torch.float32)
        y = tb.Y[sl].to(device, non_blocking=True)
        lm = tb.LM[sl].to(device, non_blocking=True)
        yield x, m, y, lm

@torch.no_grad()
def eval_test(model):
    model.eval()
    preds, truths = [], []
    for x, m, y, lm in iter_batches(tb_te, BATCH_SIZE, False, DEVICE):
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            logits = model(x, m, active=active)
        pred_day = logits.argmax(dim=1).cpu().numpy()
        truth_day = y.argmax(dim=1).cpu().numpy()
        truth_day[~lm.cpu().numpy()] = -1
        preds.append(pred_day); truths.append(truth_day)
    return np.concatenate(preds), np.concatenate(truths)

# Build model
model = MultiStreamLSTM(feature_names=train_b['feature_names'],
                          hidden=HIDDEN, num_layers=1, n_phases=7, dropout=0.1).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))
rng = torch.Generator().manual_seed(0)

print(f'Stream: {STREAM}, hidden={HIDDEN}, batch={BATCH_SIZE}, lr={LR}')
print(f'Training to {TOTAL_EPOCHS} epochs, eval at {CHECKPOINT_EPOCHS}')
print()

history = []

t0 = time.time()
for ep in range(1, TOTAL_EPOCHS + 1):
    model.train()
    total, nb = 0.0, 0
    for x, m, y, lm in iter_batches(tb_tr, BATCH_SIZE, True, DEVICE, rng):
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            logits = model(x, m, active=active)
            loss = soft_label_loss(logits, y, lm)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        total += loss.item(); nb += 1
    avg_loss = total / max(nb, 1)

    if ep in CHECKPOINT_EPOCHS:
        pred, truth = eval_test(model)
        rep = evaluate_predictions(pred_days=pred.astype(np.float32),
                                    true_days=truth.astype(np.float32),
                                    phase_ids=phase_ids,
                                    years=tb_te.yr.numpy(),
                                    ranks=tb_te.rank.numpy())
        row = {'epoch': ep, 'train_loss': round(avg_loss, 4),
                'macro_mae': round(rep['headline']['macro_mae'], 2),
                'macro_r2':  round(rep['headline']['macro_r2'], 3),
                'within_5d': round(rep['headline']['macro_within_5d'], 3),
                'elapsed_s': round(time.time() - t0, 1)}
        # Add per-phase MAE
        for phase, m in rep['per_phase'].iterrows():
            row[f'mae_p{int(phase)}'] = round(m['mae'], 2)
        history.append(row)
        print(f'ep={ep:3d}  loss={avg_loss:.4f}  '
              f'macro_mae={row["macro_mae"]:.2f}  '
              f'within_5d={row["within_5d"]:.3f}  '
              f'elapsed={row["elapsed_s"]:.1f}s')

print(f'\nDone. Total time: {time.time()-t0:.1f}s')


Stream: S1+S2, hidden=32, batch=32, lr=0.001
Training to 40 epochs, eval at [5, 10, 15, 20, 25, 30, 35, 40]

ep=  5  loss=0.0622  macro_mae=8.38  within_5d=0.425  elapsed=70.9s
ep= 10  loss=0.0612  macro_mae=8.40  within_5d=0.432  elapsed=140.9s
ep= 15  loss=0.0608  macro_mae=8.11  within_5d=0.445  elapsed=208.0s
ep= 20  loss=0.0604  macro_mae=8.10  within_5d=0.444  elapsed=285.6s
ep= 25  loss=0.0602  macro_mae=8.07  within_5d=0.445  elapsed=356.8s
ep= 30  loss=0.0602  macro_mae=7.69  within_5d=0.472  elapsed=425.7s
ep= 35  loss=0.0601  macro_mae=8.09  within_5d=0.443  elapsed=504.5s
ep= 40  loss=0.0601  macro_mae=8.42  within_5d=0.421  elapsed=582.9s

Done. Total time: 582.9s


## 4. Plot learning curve

In [4]:
hist_df = pd.DataFrame(history)
print('=== Learning curve (test-set metrics at each checkpoint) ===')
print(hist_df.to_string(index=False))

print('\n=== Per-phase MAE across checkpoints ===')
mae_cols = [c for c in hist_df.columns if c.startswith('mae_p')]
print(hist_df[['epoch'] + mae_cols].to_string(index=False))


=== Learning curve (test-set metrics at each checkpoint) ===
 epoch  train_loss  macro_mae  macro_r2  within_5d  elapsed_s  mae_p10  mae_p12  mae_p15  mae_p18  mae_p19  mae_p21  mae_p24
     5      0.0622       8.38    -0.128      0.425       70.9     9.73    10.84     9.90     7.45     7.79     6.98     5.98
    10      0.0612       8.40    -0.108      0.432      140.9     9.44    10.20    10.79     7.08     7.92     7.21     6.14
    15      0.0608       8.11    -0.014      0.445      208.0     9.94    10.59     8.90     6.24     7.93     7.36     5.84
    20      0.0604       8.10    -0.022      0.444      285.6     9.70    10.27     9.14     8.00     7.61     6.73     5.28
    25      0.0602       8.07     0.005      0.445      356.8    10.31    10.62     8.66     7.06     7.60     6.84     5.42
    30      0.0602       7.69     0.094      0.472      425.7     9.49    10.27     8.39     6.11     7.60     6.77     5.18
    35      0.0601       8.09     0.019      0.443      504.5   

In [5]:
# Simple text plot (no matplotlib needed)
print('\n=== Macro MAE trajectory (lower = better) ===')
maes = hist_df['macro_mae'].values
best = maes.min()
worst = maes.max()
for ep, mae in zip(hist_df['epoch'], maes):
    bar_len = int(40 * (mae - best) / (worst - best + 1e-6))
    bar = '█' * (40 - bar_len)
    marker = '  ← best' if mae == best else ''
    print(f'  ep={ep:3d}  MAE={mae:.2f}  {bar}{marker}')



=== Macro MAE trajectory (lower = better) ===
  ep=  5  MAE=8.38  ███
  ep= 10  MAE=8.40  ██
  ep= 15  MAE=8.11  █████████████████
  ep= 20  MAE=8.10  ██████████████████
  ep= 25  MAE=8.07  ████████████████████
  ep= 30  MAE=7.69  ████████████████████████████████████████  ← best
  ep= 35  MAE=8.09  ███████████████████
  ep= 40  MAE=8.42  █


## 5. Interpretation guide

Look at the macro_mae column:

- **Monotonic improvement up to epoch 40** → we're undertrained; rerun grid with more epochs.
- **Plateau by epoch 15–20** → 15 was already enough; current Phase 3 results are fine.
- **Best around epoch 20–25, then worse at 30+** → we hit overfitting; epochs=20 is the sweet spot.

Whatever the answer, Phase 4 (MIL) will use that as its training budget.
